# Sentinel-1 Download from Copernicus (COG) 2017–2021 — 5° Tiles

Downloads Sentinel-1 (flood labels) from Copernicus CDSE using COG. **Matches MODIS 5° tiling** from `modis_only_download.ipynb`:
- **23 tiles** (5° × 5° each) × **46 periods/year** × **5 years** = 5,290 tile-date combinations
- Same grid as `Indo_5d_MODIS_*` – tiles align for 64×64 patch creation
- Resampled to 500m (MODIS scale) for alignment
- Output: `Indo_5d_S1_{year}_{p_idx:03d}_Tile_{i}.npy` (matches `Indo_5d_MODIS_{year}_{p_idx:03d}_Tile_{i}`)

**64×64 patching** for CNN-LSTM is done in the datamodule or the section at the end.

In [7]:
!pip install -q shapely earthengine-api

In [8]:
# Setup: Mount Drive, install deps, CDSE credentials
from google.colab import drive
drive.mount('/content/drive')

!pip install -q boto3 pyproj pystac-client rioxarray xarray numpy
%cd /content/drive/MyDrive/indonesia-flood-cnn-lstm
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import importlib
import pipeline.downloader
importlib.reload(pipeline.downloader)
from pipeline.downloader import set_cdse_credentials, fetch_data_stac, download_and_load_assets

try:
    from google.colab import userdata
    set_cdse_credentials(
        client_id=userdata.get('CDSE_CLIENT_ID'),
        client_secret=userdata.get('CDSE_CLIENT_SECRET'),
        s3_access_key=userdata.get('CDSE_S3_ACCESS_KEY'),
        s3_secret_key=userdata.get('CDSE_S3_SECRET_KEY'),
    )
    cdse_creds = None
    print('✓ CDSE credentials set')
except Exception as e:
    cdse_creds = None
    print('CDSE missing:', e)

OUTPUT_DIR = Path('/content/drive/MyDrive/indonesia-flood-cnn-lstm/data/s1_labels_5d')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/indonesia-flood-cnn-lstm
✓ CDSE credentials set


In [9]:
# Tile bboxes: SAME as modis_only_download.ipynb (5° × 5°, Indonesia land only)
# Must match Indo_5d_MODIS_* exactly for correct alignment
import ee
from google.colab import auth
auth.authenticate_user()
ee.Initialize(project='flooding-process')

countries = ee.FeatureCollection('FAO/GAUL/2015/level0')
indonesia_land = countries.filter(ee.Filter.eq('ADM0_NAME', 'Indonesia')).geometry()
bbox = ee.Geometry.Rectangle([95.0, -11.0, 126.0, 6.0])
indonesia_region = indonesia_land.intersection(bbox)
grid_size = 5.0

def get_grid_cells(roi, grid_size):
    """Same as modis_only_download.ipynb – keep in sync."""
    bounds = roi.bounds().getInfo()['coordinates'][0]
    min_x, max_x = min(p[0] for p in bounds), max(p[0] for p in bounds)
    min_y, max_y = min(p[1] for p in bounds), max(p[1] for p in bounds)
    features = []
    x = min_x
    while x < max_x:
        y = min_y
        while y < max_y:
            geom = ee.Geometry.Rectangle([x, y, min(x + grid_size, max_x), min(y + grid_size, max_y)])
            features.append(ee.Feature(geom))
            y += grid_size
        x += grid_size
    fc = ee.FeatureCollection(features)
    filtered = fc.filterBounds(roi)
    info = filtered.getInfo()
    return [ee.Geometry(f['geometry']) for f in info.get('features', [])]

tiles = get_grid_cells(indonesia_region, grid_size)
tiles_to_use = tiles
tile_indices = list(range(len(tiles)))

def geom_to_bbox(geom):
    info = geom.getInfo()
    coords = info['coordinates'][0]
    xs, ys = [c[0] for c in coords], [c[1] for c in coords]
    return [min(xs), min(ys), max(xs), max(ys)]

print(f'Tiles: {len(tiles_to_use)} (5° × 5° each, matches Indo_5d_MODIS)')

Tiles: 23 (5° × 5° each, matches Indo_5d_MODIS)


In [10]:
# 8-day periods: SAME as modis_only_download (year + p_idx: 000, 001, … 045)
from datetime import datetime, timedelta

def get_8day_periods(year):
    """Same as modis_only_download – returns (start_date, end_date) per period."""
    d = datetime(year, 1, 1)
    periods = []
    while d.year == year:
        end = min(d + timedelta(days=7), datetime(year, 12, 31))
        periods.append((d.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')))
        d += timedelta(days=8)
    return periods

YEARS = range(2017, 2022)
s1_periods = [(year, p_idx, start_d) for year in YEARS for p_idx, (start_d, _) in enumerate(get_8day_periods(year))]
print(f'Periods: {len(s1_periods)} (matches MODIS year + p_idx naming)')

Periods: 230 (matches MODIS year + p_idx naming)


In [14]:
# Download S1 (COG). 500m in-read resampling ≈ 4GB/tile – safe for 8 parallel workers.
import gc
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_one(args):
    tile_id, bbox, date, out_path = args
    if out_path.exists():
        return ('skipped', tile_id, date)
    try:
        items = fetch_data_stac(bbox, date, s1_days_window=14, use_copernicus_s1=True, cdse_credentials=None, s1_only=True)
        if not items.get('s1'):
            return ('no_s1', tile_id, date)
        data = download_and_load_assets(items, bbox, cdse_credentials=None, s1_only=True, s1_target_resolution_m=500)
        arr = np.asarray(data['s1'].squeeze())
        np.save(out_path, arr)
        del data, arr
        gc.collect()
        return ('saved', tile_id, date)
    except Exception as e:
        return ('error', tile_id, date, str(e))

tasks = []
for i, tile_roi in enumerate(tiles_to_use):
    bbox = geom_to_bbox(tile_roi)
    for year, p_idx, start_d in s1_periods:
        out_path = OUTPUT_DIR / f'Indo_5d_S1_{year}_{p_idx:03d}_Tile_{tile_indices[i]}.npy'
        tasks.append((tile_indices[i], bbox, start_d, out_path))

MAX_WORKERS = 8
saved, skipped, no_s1, errors = 0, 0, 0, 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(fetch_one, t): t for t in tasks}
    for i, fut in enumerate(as_completed(futures)):
        res = fut.result()
        if res[0] == 'saved': saved += 1
        elif res[0] == 'skipped': skipped += 1
        elif res[0] == 'no_s1': no_s1 += 1
        else: errors += 1
        if (i + 1) % 200 == 0:
            print(f'  Progress: {i+1}/{len(tasks)} | saved={saved} skipped={skipped} no_s1={no_s1} errors={errors}')
            gc.collect()

print(f'Done: saved={saved} skipped={skipped} no_s1={no_s1} errors={errors}')

  Progress: 200/5290 | saved=0 skipped=200 no_s1=0 errors=0
  Progress: 400/5290 | saved=0 skipped=400 no_s1=0 errors=0
  Progress: 600/5290 | saved=0 skipped=600 no_s1=0 errors=0
  Progress: 800/5290 | saved=0 skipped=800 no_s1=0 errors=0
  Progress: 1000/5290 | saved=0 skipped=1000 no_s1=0 errors=0
  Progress: 1200/5290 | saved=0 skipped=1200 no_s1=0 errors=0
  Progress: 1400/5290 | saved=0 skipped=1400 no_s1=0 errors=0
  Progress: 1600/5290 | saved=0 skipped=1600 no_s1=0 errors=0
  Progress: 1800/5290 | saved=0 skipped=1800 no_s1=0 errors=0


  Progress: 2000/5290 | saved=0 skipped=1994 no_s1=6 errors=0
  Progress: 2200/5290 | saved=0 skipped=2194 no_s1=6 errors=0
  Progress: 2400/5290 | saved=0 skipped=2394 no_s1=6 errors=0
  Progress: 2600/5290 | saved=0 skipped=2594 no_s1=6 errors=0
  Progress: 2800/5290 | saved=0 skipped=2794 no_s1=6 errors=0
  Progress: 3000/5290 | saved=0 skipped=2994 no_s1=6 errors=0
  Progress: 3200/5290 | saved=0 skipped=3194 no_s1=6 errors=0
  Progress: 3400/5290 | saved=0 skipped=3394 no_s1=6 errors=0
  Progress: 3600/5290 | saved=0 skipped=3594 no_s1=6 errors=0
  Progress: 3800/5290 | saved=0 skipped=3794 no_s1=6 errors=0
  Progress: 4000/5290 | saved=0 skipped=3994 no_s1=6 errors=0
  Progress: 4200/5290 | saved=0 skipped=4194 no_s1=6 errors=0
  Progress: 4400/5290 | saved=0 skipped=4394 no_s1=6 errors=0
  Progress: 4600/5290 | saved=0 skipped=4594 no_s1=6 errors=0
  Progress: 4800/5290 | saved=0 skipped=4794 no_s1=6 errors=0
  Progress: 5000/5290 | saved=0 skipped=4994 no_s1=6 errors=0
  Progre

Done: saved=0 skipped=5283 no_s1=7 errors=0


# 64×64 Patch Creation for CNN-LSTM

Your model expects **64×64 pixel patches**. The pipeline's `create_patches` in `processor.py` does this during dataset build. You can either:

1. Set `patch_size: 64` in `config.yaml` (instead of 32)
2. Use the datamodule as-is – it loads tile-level MODIS/S1/DEM and creates patches on the fly

If you need to **pre-split** tiles into 64×64 and save for faster training:

In [ ]:
# Optional: pre-create 64×64 patches from tile-level data
# Run after S1 download. Assumes MODIS and DEM are in the same tile layout.

import numpy as np
from pathlib import Path

def create_patches(arr, patch_size=64, stride=64):
    """Split array into non-overlapping patches."""
    if arr.ndim == 2:
        h, w = arr.shape
        patches = []
        for y in range(0, h - patch_size + 1, stride):
            for x in range(0, w - patch_size + 1, stride):
                patches.append(arr[y:y+patch_size, x:x+patch_size])
        return np.stack(patches) if patches else np.empty((0, patch_size, patch_size))
    elif arr.ndim == 3:
        # (bands, h, w) or (t, h, w)
        _, h, w = arr.shape
        patches = []
        for y in range(0, h - patch_size + 1, stride):
            for x in range(0, w - patch_size + 1, stride):
                patches.append(arr[:, y:y+patch_size, x:x+patch_size])
        return np.stack(patches) if patches else np.empty((0, arr.shape[0], patch_size, patch_size))
    return arr

PATCH_SIZE = 64
STRIDE = 64  # No overlap; use 32 for 50% overlap

# Example: load one S1 tile, create patches (period 000 = Jan 1, matches Indo_5d_MODIS_2017_000)
example_path = OUTPUT_DIR / 'Indo_5d_S1_2017_000_Tile_0.npy'
if example_path.exists():
    s1 = np.load(example_path)
    patches = create_patches(s1.squeeze(), patch_size=PATCH_SIZE, stride=STRIDE)
    print(f'S1 shape: {s1.shape} -> {patches.shape[0]} patches of {PATCH_SIZE}×{PATCH_SIZE}')
else:
    print('Run S1 download first. Example file not found.')

S1 shape: (1114, 1114) -> 289 patches of 64×64
